In [2]:
import os, requests, zipfile
from google.colab import userdata

token = userdata.get('KAGGLE_TOKEN')
headers = {"Authorization": f"Bearer {token}"}

print("Downloading...")
r = requests.get(
    "https://www.kaggle.com/api/v1/datasets/download/abhisheksjha/time-series-air-quality-data-of-india-2010-2023",
    headers=headers,
    stream=True
)

with open('/content/air_quality.zip', 'wb') as f:
    for chunk in r.iter_content(chunk_size=8192):
        f.write(chunk)

os.makedirs('/content/air_quality', exist_ok=True)
with zipfile.ZipFile('/content/air_quality.zip', 'r') as z:
    z.extractall('/content/air_quality')

print("Done!")

Downloading...
Done!


In [4]:
import pandas as pd
import numpy as np

In [5]:
final_files = {
    'Delhi':     'DL030',
    'Mumbai':    'MH015',
    'Chennai':   'TN003',
    'Hyderabad': 'TG004',
    'Bengaluru': 'KA008',
}

kaggle_dfs = {}

for city, fname in final_files.items():
    df = pd.read_csv(f'/content/air_quality/{fname}.csv')
    pm25_col = [c for c in df.columns if 'PM2.5' in c][0]
    df = df[['From Date', pm25_col]].copy()
    df.columns = ['date', 'pm25']
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df = df.dropna(subset=['date']).set_index('date')
    daily = df['pm25'].resample('D').mean().reset_index()
    daily.columns = ['date', 'pm25']
    daily = daily[daily['date'] >= '2019-01-01'].reset_index(drop=True)
    daily['pm25'] = daily['pm25'].interpolate(method='linear')
    kaggle_dfs[city] = daily

for city, df in kaggle_dfs.items():
    print(f"{city}: {len(df)} days | nulls: {df['pm25'].isna().sum()}")

Delhi: 1551 days | nulls: 0
Mumbai: 1381 days | nulls: 0
Chennai: 1551 days | nulls: 0
Hyderabad: 1551 days | nulls: 0
Bengaluru: 1551 days | nulls: 0


In [6]:
import numpy as np

for city, df in kaggle_dfs.items():
    df['pm25_log'] = np.log1p(df['pm25'])

kaggle_dfs['Delhi'].head()

,date,pm25,pm25_log
0,2019-01-01,144.833333,4.982464
1,2019-01-02,221.937500,5.406891
2,2019-01-03,320.093750,5.771733
3,2019-01-04,265.604167,5.585765
4,2019-01-05,273.427083,5.614686


In [7]:
lags = [1, 2, 3, 7, 14, 30, 365]

for city, df in kaggle_dfs.items():
    for lag in lags:
        df[f'lag_{lag}'] = df['pm25_log'].shift(lag)

kaggle_dfs['Delhi'].head()

,date,pm25,pm25_log,lag_1,lag_2,lag_3,lag_7,lag_14,lag_30,lag_365
0,2019-01-01,144.833333,4.982464,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2019-01-02,221.937500,5.406891,4.982464,NaN,NaN,NaN,NaN,NaN,NaN
2,2019-01-03,320.093750,5.771733,5.406891,4.982464,NaN,NaN,NaN,NaN,NaN
3,2019-01-04,265.604167,5.585765,5.771733,5.406891,4.982464,NaN,NaN,NaN,NaN
4,2019-01-05,273.427083,5.614686,5.585765,5.771733,5.406891,NaN,NaN,NaN,NaN


In [8]:
windows = [7, 14, 30]

for city, df in kaggle_dfs.items():
    shifted = df['pm25_log'].shift(1)
    for w in windows:
        df[f'roll_mean_{w}'] = shifted.rolling(w).mean()
        df[f'roll_std_{w}'] = shifted.rolling(w).std()

kaggle_dfs['Delhi'].head(10)

,date,pm25,pm25_log,lag_1,lag_2,lag_3,lag_7,lag_14,lag_30,lag_365,roll_mean_7,roll_std_7,roll_mean_14,roll_std_14,roll_mean_30,roll_std_30
0,2019-01-01,144.833333,4.982464,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2019-01-02,221.937500,5.406891,4.982464,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2019-01-03,320.093750,5.771733,5.406891,4.982464,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2019-01-04,265.604167,5.585765,5.771733,5.406891,4.982464,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2019-01-05,273.427083,5.614686,5.585765,5.771733,5.406891,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2019-01-06,169.395833,5.138124,5.614686,5.585765,5.771733,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2019-01-07,167.440833,5.126585,5.138124,5.614686,5.585765,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2019-01-08,169.385417,5.138063,5.126585,5.138124,5.614686,4.982464,NaN,NaN,NaN,5.375178,0.297823,NaN,NaN,NaN,NaN
8,2019-01-09,165.562500,5.115371,5.138063,5.126585,5.138124,5.406891,NaN,NaN,NaN,5.397407,0.267934,NaN,NaN,NaN,NaN
9,2019-01-10,184.572917,5.223448,5.115371,5.138063,5.126585,5.771733,NaN,NaN,NaN,5.355761,0.288110,NaN,NaN,NaN,NaN


In [9]:
for city, df in kaggle_dfs.items():
    shifted = df['pm25_log'].shift(1)
    df['roll_min_7'] = shifted.rolling(7).min()
    df['roll_max_7'] = shifted.rolling(7).max()
    df['roll_min_30'] = shifted.rolling(30).min()
    df['roll_max_30'] = shifted.rolling(30).max()

kaggle_dfs['Delhi'].head(10)

,date,pm25,pm25_log,lag_1,lag_2,lag_3,lag_7,lag_14,lag_30,lag_365,roll_mean_7,roll_std_7,roll_mean_14,roll_std_14,roll_mean_30,roll_std_30,roll_min_7,roll_max_7,roll_min_30,roll_max_30
0,2019-01-01,144.833333,4.982464,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2019-01-02,221.937500,5.406891,4.982464,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2019-01-03,320.093750,5.771733,5.406891,4.982464,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2019-01-04,265.604167,5.585765,5.771733,5.406891,4.982464,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2019-01-05,273.427083,5.614686,5.585765,5.771733,5.406891,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2019-01-06,169.395833,5.138124,5.614686,5.585765,5.771733,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2019-01-07,167.440833,5.126585,5.138124,5.614686,5.585765,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2019-01-08,169.385417,5.138063,5.126585,5.138124,5.614686,4.982464,NaN,NaN,NaN,5.375178,0.297823,NaN,NaN,NaN,NaN,4.982464,5.771733,NaN,NaN
8,2019-01-09,165.562500,5.115371,5.138063,5.126585,5.138124,5.406891,NaN,NaN,NaN,5.397407,0.267934,NaN,NaN,NaN,NaN,5.126585,5.771733,NaN,NaN
9,2019-01-10,184.572917,5.223448,5.115371,5.138063,5.126585,5.771733,NaN,NaN,NaN,5.355761,0.288110,NaN,NaN,NaN,NaN,5.115371,5.771733,NaN,NaN


In [10]:
for city, df in kaggle_dfs.items():
    df['day_of_week'] = df['date'].dt.dayofweek
    df['month'] = df['date'].dt.month
    df['day_of_year'] = df['date'].dt.dayofyear
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

kaggle_dfs['Delhi'].head()

,date,pm25,pm25_log,lag_1,lag_2,lag_3,lag_7,lag_14,lag_30,lag_365,...,roll_mean_30,roll_std_30,roll_min_7,roll_max_7,roll_min_30,roll_max_30,day_of_week,month,day_of_year,is_weekend
0,2019-01-01,144.833333,4.982464,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1,1,1,0
1,2019-01-02,221.937500,5.406891,4.982464,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,2,1,2,0
2,2019-01-03,320.093750,5.771733,5.406891,4.982464,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,3,1,3,0
3,2019-01-04,265.604167,5.585765,5.771733,5.406891,4.982464,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,4,1,4,0
4,2019-01-05,273.427083,5.614686,5.585765,5.771733,5.406891,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,5,1,5,1


In [11]:
for city, df in kaggle_dfs.items():
    df['doy_sin'] = np.sin(2 * np.pi * df['day_of_year'] / 365.25)
    df['doy_cos'] = np.cos(2 * np.pi * df['day_of_year'] / 365.25)
    df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)

kaggle_dfs['Delhi'].head()

,date,pm25,pm25_log,lag_1,lag_2,lag_3,lag_7,lag_14,lag_30,lag_365,...,roll_min_30,roll_max_30,day_of_week,month,day_of_year,is_weekend,doy_sin,doy_cos,dow_sin,dow_cos
0,2019-01-01,144.833333,4.982464,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,1,1,1,0,0.017202,0.999852,0.781831,0.623490
1,2019-01-02,221.937500,5.406891,4.982464,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,2,1,2,0,0.034398,0.999408,0.974928,-0.222521
2,2019-01-03,320.093750,5.771733,5.406891,4.982464,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,3,1,3,0,0.051584,0.998669,0.433884,-0.900969
3,2019-01-04,265.604167,5.585765,5.771733,5.406891,4.982464,NaN,NaN,NaN,NaN,...,NaN,NaN,4,1,4,0,0.068755,0.997634,-0.433884,-0.900969
4,2019-01-05,273.427083,5.614686,5.585765,5.771733,5.406891,NaN,NaN,NaN,NaN,...,NaN,NaN,5,1,5,1,0.085906,0.996303,-0.974928,-0.222521


In [12]:
holidays = pd.DataFrame({
    'holiday': [
        'Diwali', 'Diwali', 'Diwali', 'Diwali', 'Diwali',
        'Republic Day', 'Republic Day', 'Republic Day', 'Republic Day', 'Republic Day',
        'Holi', 'Holi', 'Holi', 'Holi', 'Holi',
        'New Year', 'New Year', 'New Year', 'New Year', 'New Year',
    ],
    'ds': pd.to_datetime([
        '2019-10-27', '2020-11-14', '2021-11-04', '2022-10-24', '2023-11-12',
        '2019-01-26', '2020-01-26', '2021-01-26', '2022-01-26', '2023-01-26',
        '2019-03-21', '2020-03-10', '2021-03-29', '2022-03-18', '2023-03-08',
        '2019-01-01', '2020-01-01', '2021-01-01', '2022-01-01', '2023-01-01',
    ])
})

holidays

,holiday,ds
0,Diwali,2019-10-27
1,Diwali,2020-11-14
2,Diwali,2021-11-04
3,Diwali,2022-10-24
4,Diwali,2023-11-12
5,Republic Day,2019-01-26
6,Republic Day,2020-01-26
7,Republic Day,2021-01-26
8,Republic Day,2022-01-26
9,Republic Day,2023-01-26


In [13]:
diwali_dates = pd.to_datetime([
    '2019-10-26', '2019-10-27', '2019-10-28',
    '2020-11-13', '2020-11-14', '2020-11-15',
    '2021-11-03', '2021-11-04', '2021-11-05',
    '2022-10-23', '2022-10-24', '2022-10-25',
    '2023-11-11', '2023-11-12', '2023-11-13',
])

republic_day_dates = pd.to_datetime([
    '2019-01-25', '2019-01-26', '2019-01-27',
    '2020-01-25', '2020-01-26', '2020-01-27',
    '2021-01-25', '2021-01-26', '2021-01-27',
    '2022-01-25', '2022-01-26', '2022-01-27',
    '2023-01-25', '2023-01-26', '2023-01-27',
])

holi_dates = pd.to_datetime([
    '2019-03-20', '2019-03-21', '2019-03-22',
    '2020-03-09', '2020-03-10', '2020-03-11',
    '2021-03-28', '2021-03-29', '2021-03-30',
    '2022-03-17', '2022-03-18', '2022-03-19',
    '2023-03-07', '2023-03-08', '2023-03-09',
])

new_year_dates = pd.to_datetime([
    '2018-12-31', '2019-01-01', '2019-01-02',
    '2019-12-31', '2020-01-01', '2020-01-02',
    '2020-12-31', '2021-01-01', '2021-01-02',
    '2021-12-31', '2022-01-01', '2022-01-02',
    '2022-12-31', '2023-01-01', '2023-01-02',
])

for city, df in kaggle_dfs.items():
    df['is_diwali'] = df['date'].isin(diwali_dates).astype(int)
    df['is_republic_day'] = df['date'].isin(republic_day_dates).astype(int)
    df['is_holi'] = df['date'].isin(holi_dates).astype(int)
    df['is_new_year'] = df['date'].isin(new_year_dates).astype(int)

kaggle_dfs['Delhi'].filter(like='is_').head(30)

,is_weekend,is_diwali,is_republic_day,is_holi,is_new_year
0,0,0,0,0,1
1,0,0,0,0,1
2,0,0,0,0,0
3,0,0,0,0,0
4,1,0,0,0,0
5,1,0,0,0,0
6,0,0,0,0,0
7,0,0,0,0,0
8,0,0,0,0,0
9,0,0,0,0,0


In [14]:
for city, df in kaggle_dfs.items():
    mean_log = df['pm25_log'].mean()
    std_log = df['pm25_log'].std()
    z_score = (df['pm25_log'] - mean_log) / std_log
    df['is_outlier'] = (z_score.abs() > 3).astype(int)

for city, df in kaggle_dfs.items():
    n_outliers = df['is_outlier'].sum()
    print(f"{city}: {n_outliers} outlier days flagged")

Delhi: 0 outlier days flagged
Mumbai: 0 outlier days flagged
Chennai: 8 outlier days flagged
Hyderabad: 7 outlier days flagged
Bengaluru: 4 outlier days flagged


In [15]:
bengaluru_check = kaggle_dfs['Bengaluru'][kaggle_dfs['Bengaluru']['date'] == '2023-01-26']
print(bengaluru_check[['date', 'pm25', 'pm25_log', 'is_outlier']])

chennai_check = kaggle_dfs['Chennai'][kaggle_dfs['Chennai']['date'] == '2019-07-05']
print(chennai_check[['date', 'pm25', 'pm25_log', 'is_outlier']])

           date   pm25  pm25_log  is_outlier
1486 2023-01-26  556.0  6.322565           1
          date        pm25  pm25_log  is_outlier
185 2019-07-05  291.490588  5.678432           1


In [16]:
import os

os.makedirs('/content/processed', exist_ok=True)

for city, df in kaggle_dfs.items():
    df.to_csv(f'/content/processed/{city}_features.csv', index=False)

In [17]:
!zip -r /content/processed_features.zip /content/processed

  adding: content/processed/ (stored 0%)
  adding: content/processed/Chennai_features.csv (deflated 73%)
  adding: content/processed/Delhi_features.csv (deflated 73%)
  adding: content/processed/Hyderabad_features.csv (deflated 73%)
  adding: content/processed/Mumbai_features.csv (deflated 73%)
  adding: content/processed/Bengaluru_features.csv (deflated 73%)


In [18]:
from google.colab import files
files.download('/content/processed_features.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>